In [2]:
import os, sys
import traceback
import pandas as pd
import numpy as np
import random
import copy
import time
import pickle
import torch
import torch.nn as nn
from bert.atom_embedding_generator import bert_atom_embedding

In [5]:
kbert_weight_path = "pretrain_k_bert_epoch_7.pth"
csv_path = "demo/example_data/example_data.csv"
atom_pkl_path = "atom_features_dict.pkl"
global_csv_path = "example_data_kbert_out.csv"

In [1]:
def save_pickle(obj, save_path):
    with open(save_path, "wb") as f:
        pickle.dump(obj, f)

def load_pickle(load_path):
    with open(load_path, "rb") as f:
        data = pickle.load(f)
    return data

In [3]:
def drug_id2encoding(smiles_list):
    drug_atom_list = []
    drug_global_list = []
    for i, smiles in enumerate(smiles_list):
        try:
            h_global, g_atom = bert_atom_embedding(smiles, pretrain_model=kbert_weight_path)
            drug_atom_list.append(g_atom)
            drug_global_list.append(h_global)
        except Exception as e:
            print(f"Encoding failed number{i} SMILES:{smiles}, error:{str(e)}")
            print("Complete error stack trace：")
            print(traceback.format_exc())

            fail_atom = np.full((1, 768), np.nan, dtype=np.float32)
            fail_global = np.full((768,), np.nan, dtype=np.float32)
            drug_atom_list.append(fail_atom)
            drug_global_list.append(fail_global)
    return drug_atom_list, drug_global_list

In [6]:
def generate_kbert_offline_features():
    df = pd.read_csv(csv_path)
    all_smiles = df['smiles'].tolist()
    print(f"Total number of molecules：{len(all_smiles)}")
    atom_list, global_list = drug_id2encoding(all_smiles)

    atom_dict = {}
    for smi, atom_mat in zip(all_smiles, atom_list):
       atom_dict[smi] = atom_mat
    save_pickle(atom_dict, atom_pkl_path)
    print(f"Atomic feature dictionary saved successfully：{atom_pkl_path}")

    rows = []
    for smi, vec in zip(all_smiles, global_list):
        row = {"smiles": smi}
        for idx in range(768):
            row[f"pretrain_feature_{idx+1}"] = vec[idx]
        rows.append(row)
    df_global = pd.DataFrame(rows)
    df_global.to_csv(global_csv_path, index=False)
    print(f"Molecular feature preservation completed：{global_csv_path}")
df_global = pd.read_csv(global_csv_path)
print(f"feature shape: {df_global.values[:, 1:].shape}")

feature shape: (9, 768)
